# Step 2 — TF-IDF features

Step 1 left a segmented, UTF-8 corpus in `../data/processed/`. This notebook turns it into the
feature matrix the network trains on.

**This step does not use `layers.TextVectorization`, and the reason is worth stating.** That layer
exists to put preprocessing *inside* the model, so a deployed model takes raw strings and there is
no separate Python step that can drift out of sync with training. Here that guarantee is
unreachable: `underthesea` is a Python CRF model and cannot run inside a TensorFlow graph, so a
Python preprocessing step outside the model is mandatory whatever Step 2 does. The project would be
paying every cost of `TextVectorization` — dense output, a standardizer that deletes `_`, no
`min_df`, an unreadable vocabulary — for a benefit it structurally cannot collect.

sklearn's `TfidfVectorizer` gives sparse output, leaves `_` alone, has `min_df`, and returns
readable feature names. Keras is unaffected: the model in Step 3 is still Keras, and it accepts a
scipy sparse matrix directly. See `../Personal Note.md` for the full reasoning and measurements.

## The tokenizer

Three jobs, and each is there because of something measured rather than assumed:

- **strip punctuation, keep `_`** — 13.5% of tokens in the segmented corpus are bare punctuation
  (`.`, `,`, `"`). `tokenizer=str.split` alone would let every one of them become a feature.
- **`_` must survive** — it is what marks a compound word. Note that `string.punctuation`
  *contains* `_`, which is exactly the trap that has to be avoided here.
- **drop tokens containing digits** — 43,648 numbers in a 3,000-article sample. Dates and figures
  do not identify a topic, and each distinct one would otherwise occupy a vocabulary slot.

In [1]:
import re, string
PUNCT  = (set(string.punctuation) | set("\u201c\u201d\u2018\u2019\u2026\u2013\u2014\u2022\u00b7")) - {"_"}
_DIGIT = re.compile(r"\d")

def vn_tokenizer(doc):
    """Tach theo khoang trang, bo dau cau, GIU '_', bo token chua chu so."""
    out = []
    for t in doc.split():
        t = "".join(c for c in t if c not in PUNCT).strip("_")
        if t and not _DIGIT.search(t):
            out.append(t)
    return out

# quick check on the cases that matter
for t in ["xu\u1ea5t_kh\u1ea9u", ".", '"x\u01b0\u01a1ng"', "17/4", "TP.HCM", "!_Anh", "USD"]:
    print(f"{t!r:12} -> {vn_tokenizer(t)}")

'xuất_khẩu'  -> ['xuất_khẩu']
'.'          -> []
'"xương"'    -> ['xương']
'17/4'       -> []
'TP.HCM'     -> ['TPHCM']
'!_Anh'      -> ['Anh']
'USD'        -> ['USD']


## Load the corpus

`sorted()` here is deliberate and also dangerous — it groups files by class folder, which is fine
for loading but means the arrays arrive **sorted by label**. That matters in the next cell.

In [2]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

def load_split(split):
    paths  = sorted(Path(f"../data/processed/{split}").rglob("*.txt"))
    texts  = [p.read_text(encoding="utf-8") for p in paths]
    labels = [p.parent.name for p in paths]
    return texts, labels

train_texts, train_labels = load_split("Train_Full")
test_texts,  test_labels  = load_split("Test_Full")

le = LabelEncoder().fit(train_labels)
y_train, y_test = le.transform(train_labels), le.transform(test_labels)
class_names = list(le.classes_)

print(f"train {len(train_texts):,}   test {len(test_texts):,}")
print("classes:", class_names)

train 33,759   test 50,373
classes: ['Chinh tri Xa hoi', 'Doi song', 'Khoa hoc', 'Kinh doanh', 'Phap luat', 'Suc khoe', 'The gioi', 'The thao', 'Van hoa', 'Vi tinh']


## Fit TF-IDF — on the training set only

`fit_transform` on train, plain `transform` on test. The vocabulary and the IDF weights are learned
from training data and then *applied* to test. Calling `fit` on the test set would let the model
see which words are rare in the data it is about to be scored on — leakage, and a silent one.

`token_pattern=None` is required when passing a custom `tokenizer`, otherwise sklearn warns that
`token_pattern` is being ignored.

`max_features=10000, min_df=3` come from a validation sweep (see Step 3): 10k vs 20k features and
`min_df` 3 vs 5 all landed within 0.001 macro-F1 of each other, so the smallest was kept.

In [3]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vectorizer = TfidfVectorizer(
    tokenizer=vn_tokenizer,
    lowercase=True,          # .lower() is diacritic-safe: XU\u1ea4T_KH\u1ea8U -> xu\u1ea5t_kh\u1ea9u
    token_pattern=None,
    strip_accents=None,      # MUST stay None -- see markdown below
    max_features=10000,
    min_df=3,
)
X_train = vectorizer.fit_transform(train_texts).astype("float32")
X_test  = vectorizer.transform(test_texts).astype("float32")
print(f"fitted in {time.time()-t0:.0f}s")
print("X_train", X_train.shape, " X_test", X_test.shape)

fitted in 81s
X_train (33759, 10000)  X_test (50373, 10000)


### `strip_accents` must stay `None`

`TfidfVectorizer` has a `strip_accents` parameter that defaults to `None`. It exists for languages
where accents are decorative. Vietnamese is not one of them — `ma / má / mà / mã / mạ / mả` are six
different words. Setting `strip_accents="unicode"` turns `Việt_Nam xuất_khẩu` into
`viet_nam xuat_khau` and collapses them all into one token.

## Why sparse is the whole ballgame

TF-IDF produces one row per document with one column per vocabulary entry. Almost every cell is
zero — an article uses a couple of hundred distinct words out of 10,000.

In [4]:
dense_mb  = X_train.shape[0] * X_train.shape[1] * 4 / 1e6
sparse_mb = (X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes) / 1e6
print(f"non-zero cells : {X_train.nnz:,} of {X_train.shape[0]*X_train.shape[1]:,} "
      f"({100*X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.2f}% -> {100-100*X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.1f}% empty)")
print(f"distinct words per article: {X_train.nnz/X_train.shape[0]:.0f}")
print(f"as dense float32 : {dense_mb:8.1f} MB")
print(f"as sparse CSR    : {sparse_mb:8.1f} MB   ({dense_mb/sparse_mb:.0f}x smaller)")

non-zero cells : 5,561,161 of 337,590,000 (1.65% -> 98.4% empty)
distinct words per article: 165
as dense float32 :   1350.4 MB
as sparse CSR    :     44.6 MB   (30x smaller)


That ratio is why the tutorial's advice to `.cache()` everything is wrong here: cached dense, this
matrix alone would be over a gigabyte, and the RAM problem would look inherent to the task when it
is really a consequence of choosing a representation that forces dense output.

## What the vectorizer learned

In [5]:
import numpy as np

names = vectorizer.get_feature_names_out()
compounds = [n for n in names if "_" in n]
print(f"{len(compounds):,}/{len(names):,} features are compound words "
      f"({100*len(compounds)/len(names):.1f}%)")
print("examples:", compounds[3000:3008].tolist() if hasattr(compounds, 'tolist') else compounds[3000:3008])
print("longest :", sorted(names, key=lambda n: -n.count("_"))[:3])
print("features still containing punctuation:",
      [n for n in names if any(c in PUNCT for c in n)] or "none")

6,313/10,000 features are compound words (63.1%)
examples: ['người_lao_động', 'người_lớn', 'người_mẫu', 'người_nhà', 'người_phát_ngôn', 'người_quản_lý', 'người_ta', 'người_thân']
longest : ['bộ_giáo_dục_và_đào_tạo', 'bộ_luật_tố_tụng_hình_sự', 'phương_tiện_thông_tin_đại_chúng']
features still containing punctuation: none


Roughly 60% of the feature space consists of compounds that would not exist without the
segmentation step. Note carefully what that does and does not prove: it shows segmentation
*changes* most of the feature space, not that it *helps*. Step 5 tests that directly by running the
identical pipeline on the unsegmented corpus.

In [6]:
idf   = vectorizer.idf_
order = np.argsort(idf)
print("LOWEST IDF (everywhere, therefore uninformative)")
for i in order[:10]:
    print(f"   {names[i]:<16} {idf[i]:.3f}")
print("\nHIGHEST IDF (rare -- which is not the same as informative)")
for i in order[-10:]:
    print(f"   {names[i]:<16} {idf[i]:.3f}")

LOWEST IDF (everywhere, therefore uninformative)
   và               1.060
   của              1.105
   trong            1.143
   có               1.163
   là               1.168
   cho              1.172
   được             1.181
   đã               1.189
   các              1.205
   với              1.226

HIGHEST IDF (rare -- which is not the same as informative)
   rebecca          9.230
   value            9.348
   larry            9.348
   kcb              9.348
   đkhk             9.348
   đột_quị          9.481
   hoc_sinh         9.481
   kinh_dịch        9.635
   document         9.635
   contactus        10.041


Two things to read off this.

**The low end justifies skipping stopword removal.** `và`, `của`, `trong`, `là` sit at IDF ≈ 1.06
against ≈ 10.0 at the other end — a ~10× weight difference that TF-IDF applies automatically. The
three earlier projects in this repo all removed stopwords explicitly; with TF-IDF that step is
redundant.

**The high end is a warning, not a prize.** High IDF means *rare*, not *informative*. Fragments and
one-off strings collect the largest weights, which is TF-IDF's known weakness. `min_df` is the lever
against it — and the sweep in Step 3 found that raising it from 3 to 5 changed nothing measurable,
so the effect is real but small here.

## Save for the next steps

The matrices go to `../data/processed/` (intermediate data this project produced), so Steps 3–5
do not repeat the ~80 s vectorization. The vectorizer object itself is deliberately **not** pickled:
its `tokenizer` is a function defined in this notebook's `__main__`, which would not unpickle
cleanly in another process. Step 5 re-fits it, which it has to do anyway for the ablation.

In [7]:
import json
from scipy import sparse

sparse.save_npz("../data/processed/tfidf_train.npz", X_train)
sparse.save_npz("../data/processed/tfidf_test.npz",  X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)
json.dump({"class_names": class_names,
           "feature_names": names.tolist(),
           "idf": idf.tolist()},
          open("../data/processed/vocabulary.json", "w"), ensure_ascii=False)
print("saved")

saved
